# AutoAssess — CarDD Exploratory Data Analysis

Loads the CarDD COCO annotations from `data/raw/cardd/` and reports the statistics needed
to plan preprocessing, class balancing, and augmentation before training YOLOv8-seg.

This notebook is EDA-only per project convention: nothing here is reproducible/pipeline code.
All figures are saved to `reports/figures/eda/`.

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from autoassess.utils.config import load_config, resolve_paths

plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "base.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (configs/base.yaml) above " + str(start))

PROJECT_ROOT = find_project_root(Path.cwd())
cfg = resolve_paths(load_config(PROJECT_ROOT / "configs" / "base.yaml"), PROJECT_ROOT)

CARDD_ROOT = Path(cfg.data.raw_dir) / "cardd"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train2017", "val2017", "test2017"]

print(f"CarDD root: {CARDD_ROOT}")
print(f"Figures out: {FIG_DIR}")

## Locate the COCO annotation root

Handles the archive being nested (`CarDD_release/CarDD_COCO/...`) the way
`scripts/download_data.py` leaves it.

In [ ]:
def find_coco_root(cardd_dir: Path) -> Path:
    if (cardd_dir / "annotations").is_dir():
        return cardd_dir
    for candidate in cardd_dir.rglob("annotations"):
        if candidate.is_dir():
            return candidate.parent
    raise FileNotFoundError(
        f"No COCO 'annotations/' directory found under {cardd_dir}. "
        "Run `python scripts/download_data.py` first."
    )

COCO_ROOT = find_coco_root(CARDD_ROOT)
print(f"COCO root: {COCO_ROOT}")

In [ ]:
def load_split(split: str) -> dict:
    candidates = [
        COCO_ROOT / "annotations" / f"instances_{split}.json",
        COCO_ROOT / "annotations" / f"{split}.json",
    ]
    ann_file = next((p for p in candidates if p.exists()), None)
    if ann_file is None:
        raise FileNotFoundError(f"No annotation file found for split '{split}' in {COCO_ROOT / 'annotations'}")
    with ann_file.open("r", encoding="utf-8") as f:
        return json.load(f)

coco_by_split = {split: load_split(split) for split in SPLITS}
category_names = {c["id"]: c["name"] for c in next(iter(coco_by_split.values()))["categories"]}
CLASS_ORDER = [category_names[i] for i in sorted(category_names)]
print(f"Classes ({len(CLASS_ORDER)}): {CLASS_ORDER}")

## 1. Image count per split

In [ ]:
image_counts = {split: len(coco["images"]) for split, coco in coco_by_split.items()}
total_images = sum(image_counts.values())

df_images = pd.DataFrame(
    {"split": list(image_counts.keys()), "images": list(image_counts.values())}
)
df_images["fraction"] = df_images["images"] / total_images
print(df_images.to_string(index=False))
print(f"\nTotal images: {total_images}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(df_images["split"], df_images["images"], color="#4C72B0")
for i, v in enumerate(df_images["images"]):
    ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_ylabel("Image count")
ax.set_title(f"CarDD — images per split (total={total_images})")
fig.savefig(FIG_DIR / "01_images_per_split.png")
plt.show()

## 2. Instance count per class

In [ ]:
instance_counts = Counter()
for split, coco in coco_by_split.items():
    for ann in coco["annotations"]:
        instance_counts[category_names[ann["category_id"]]] += 1

df_instances = pd.DataFrame(
    {"class": CLASS_ORDER, "instances": [instance_counts[c] for c in CLASS_ORDER]}
).sort_values("instances", ascending=False)
print(df_instances.to_string(index=False))
print(f"\nTotal instances: {df_instances['instances'].sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_instances["class"], df_instances["instances"], color="#DD8452")
ax.set_ylabel("Instance count")
ax.set_title("CarDD — instance count per class (all splits)")
ax.tick_params(axis="x", rotation=30)
fig.savefig(FIG_DIR / "02_instances_per_class.png")
plt.show()

## 3. Class imbalance ratio

Ratio of the most frequent class's instance count to each class's instance count
(1.0 = majority class).

In [ ]:
max_count = df_instances["instances"].max()
df_instances["imbalance_ratio"] = max_count / df_instances["instances"]
print(df_instances.to_string(index=False))

imbalance_span = df_instances["imbalance_ratio"].max()
print(f"\nMajority:minority imbalance ratio = {imbalance_span:.1f}:1")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_instances["class"], df_instances["imbalance_ratio"], color="#C44E52")
ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("Imbalance ratio (relative to majority class)")
ax.set_title("CarDD — class imbalance ratio")
ax.tick_params(axis="x", rotation=30)
fig.savefig(FIG_DIR / "03_class_imbalance_ratio.png")
plt.show()

## 4. Mask area distribution per class (log scale)

Uses the COCO `area` field (polygon/RLE mask area in px²) per instance.

In [ ]:
areas_by_class: dict[str, list[float]] = defaultdict(list)
for coco in coco_by_split.values():
    for ann in coco["annotations"]:
        area = ann.get("area")
        if area and area > 0:
            areas_by_class[category_names[ann["category_id"]]].append(area)

area_summary = pd.DataFrame(
    {
        "class": c,
        "n": len(areas_by_class[c]),
        "median_area_px2": np.median(areas_by_class[c]) if areas_by_class[c] else np.nan,
        "p10_area_px2": np.percentile(areas_by_class[c], 10) if areas_by_class[c] else np.nan,
        "p90_area_px2": np.percentile(areas_by_class[c], 90) if areas_by_class[c] else np.nan,
    }
    for c in CLASS_ORDER
)
print(area_summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
data = [areas_by_class[c] for c in CLASS_ORDER if areas_by_class[c]]
labels = [c for c in CLASS_ORDER if areas_by_class[c]]
ax.boxplot(data, tick_labels=labels, showfliers=True)
ax.set_yscale("log")
ax.set_ylabel("Mask area (px², log scale)")
ax.set_title("CarDD — mask area distribution per class")
ax.tick_params(axis="x", rotation=30)
fig.savefig(FIG_DIR / "04_mask_area_per_class.png")
plt.show()

## 5. Image resolution distribution

In [ ]:
widths, heights = [], []
for coco in coco_by_split.values():
    for img in coco["images"]:
        widths.append(img["width"])
        heights.append(img["height"])

res_df = pd.DataFrame({"width": widths, "height": heights})
res_df["megapixels"] = res_df["width"] * res_df["height"] / 1e6
print(res_df.describe().to_string())

unique_resolutions = res_df.groupby(["width", "height"]).size().sort_values(ascending=False)
print(f"\nUnique resolutions: {len(unique_resolutions)}")
print(unique_resolutions.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(res_df["width"], res_df["height"], s=8, alpha=0.4, color="#55A868")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")
axes[0].set_title("Image resolution scatter")

axes[1].hist(res_df["megapixels"], bins=30, color="#8172B2")
axes[1].set_xlabel("Megapixels")
axes[1].set_ylabel("Image count")
axes[1].set_title("Resolution (MP) histogram")

fig.suptitle("CarDD — image resolution distribution")
fig.savefig(FIG_DIR / "05_resolution_distribution.png")
plt.show()

## 6. Instances-per-image histogram

In [ ]:
instances_per_image = Counter()
for coco in coco_by_split.values():
    for ann in coco["annotations"]:
        instances_per_image[ann["image_id"]] += 1
    # images with zero annotated instances still count as 0
    annotated_ids = {img["id"] for img in coco["images"]}
    for img_id in annotated_ids:
        instances_per_image.setdefault(img_id, 0)

counts = np.array(list(instances_per_image.values()))
print(f"mean={counts.mean():.2f}  median={np.median(counts):.1f}  "
      f"max={counts.max()}  images_with_0_instances={(counts == 0).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
max_count = int(counts.max()) if len(counts) else 0
ax.hist(counts, bins=range(0, max_count + 2), align="left", color="#64B5CD", edgecolor="white")
ax.set_xlabel("Instances per image")
ax.set_ylabel("Image count")
ax.set_title("CarDD — instances-per-image distribution")
fig.savefig(FIG_DIR / "06_instances_per_image_hist.png")
plt.show()

## Summary table (saved for reference)

In [ ]:
summary = df_instances.merge(area_summary, on="class").sort_values("instances", ascending=False)
summary.to_csv(FIG_DIR / "class_summary.csv", index=False)
summary